In [3]:
from pathlib import Path
import pandas as pd

DATA_RAW = Path("..") / "data" / "raw"

orders = pd.read_csv(DATA_RAW / "olist_orders_dataset.csv")
print("orders loaded:", orders.shape)

orders loaded: (99441, 8)


In [4]:
# The five timestamp columns in the orders table.
ORDER_DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Work on a copy so the raw DataFrame stays exactly as loaded.
orders_clean = orders.copy()

for col in ORDER_DATE_COLS:
    # errors="raise" (the default) stops with an error if any value cannot be
    # parsed. Silent coercion to NaT would hide bad data, so we avoid it here.
    orders_clean[col] = pd.to_datetime(orders_clean[col])

# Verify: missing counts must match what we measured in Phase 1.
print(orders_clean[ORDER_DATE_COLS].isna().sum().to_string())
print()
print(orders_clean[ORDER_DATE_COLS].dtypes.to_string())

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]


In [5]:
# Earliest and latest value of each timestamp column.
pd.DataFrame({
    "min": orders_clean[ORDER_DATE_COLS].min(),
    "max": orders_clean[ORDER_DATE_COLS].max(),
})

,min,max
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00


In [6]:
# A normal order should flow: purchase -> approved -> carrier -> customer.
# Comparisons involving a missing date evaluate to False, so we only compare
# rows where both dates exist.
def count_violations(earlier, later):
    both = orders_clean[earlier].notna() & orders_clean[later].notna()
    bad = both & (orders_clean[later] < orders_clean[earlier])
    return int(both.sum()), int(bad.sum())

pairs = [
    ("order_purchase_timestamp",      "order_approved_at"),
    ("order_approved_at",             "order_delivered_carrier_date"),
    ("order_delivered_carrier_date",  "order_delivered_customer_date"),
    ("order_purchase_timestamp",      "order_delivered_customer_date"),
    ("order_purchase_timestamp",      "order_estimated_delivery_date"),
]

rows = []
for earlier, later in pairs:
    compared, violations = count_violations(earlier, later)
    rows.append({"earlier": earlier, "later": later,
                 "rows_compared": compared, "later_is_before_earlier": violations})
print(pd.DataFrame(rows).to_string())

                        earlier                          later  rows_compared  later_is_before_earlier
0      order_purchase_timestamp              order_approved_at          99281                        0
1             order_approved_at   order_delivered_carrier_date          97644                     1359
2  order_delivered_carrier_date  order_delivered_customer_date          96475                       23
3      order_purchase_timestamp  order_delivered_customer_date          96476                        0
4      order_purchase_timestamp  order_estimated_delivery_date          99441                        0


In [7]:
# How large are the gaps? Positive hours = carrier pickup BEFORE approval.
gap_hours = (
    orders_clean["order_approved_at"] - orders_clean["order_delivered_carrier_date"]
).dt.total_seconds() / 3600
bad = gap_hours[gap_hours > 0]
print("approved -> carrier violations: how many hours earlier was carrier pickup?")
print(bad.describe().round(2).to_string())

# Status of the 23 'delivered before carrier pickup' orders.
both = orders_clean["order_delivered_carrier_date"].notna() & orders_clean["order_delivered_customer_date"].notna()
bad23 = both & (orders_clean["order_delivered_customer_date"] < orders_clean["order_delivered_carrier_date"])
print("\nStatus of the 23 orders:")
print(orders_clean.loc[bad23, "order_status"].value_counts().to_string())

approved -> carrier violations: how many hours earlier was carrier pickup?
count    1359.00
mean       24.75
std       115.31
min         0.01
25%         1.42
50%        17.17
75%        25.96
max      4109.26

Status of the 23 orders:
order_status
delivered    23


In [8]:
gap_days = gap_hours / 24
print("Carrier-before-approval violations by size:")
print("  over 1 day :", int((gap_days > 1).sum()))
print("  over 7 days:", int((gap_days > 7).sum()))
print("  over 30 days:", int((gap_days > 30).sum()))

Carrier-before-approval violations by size:
  over 1 day : 458
  over 7 days: 14
  over 30 days: 1


In [9]:
# Flag rows with illogical stage ordering. Comparisons with a missing date
# evaluate to False, so rows with nulls are never flagged by accident.
oc = orders_clean

oc["flag_carrier_before_approval"] = (
    oc["order_delivered_carrier_date"] < oc["order_approved_at"]
)
oc["flag_delivered_before_carrier"] = (
    oc["order_delivered_customer_date"] < oc["order_delivered_carrier_date"]
)

print(oc[["flag_carrier_before_approval", "flag_delivered_before_carrier"]].sum().to_string())

flag_carrier_before_approval     1359
flag_delivered_before_carrier      23


In [10]:
SECONDS_PER_DAY = 86400

def days_between(start, end):
    """Elapsed days from start to end (NaN if either is missing)."""
    return (end - start).dt.total_seconds() / SECONDS_PER_DAY

# Total time: purchase -> customer delivery (valid for all delivered orders
# with a delivery date; Step 2.1 found 0 negative values).
oc["delivery_days"] = days_between(
    oc["order_purchase_timestamp"], oc["order_delivered_customer_date"]
)

# Promised time: purchase -> estimated delivery date.
oc["estimated_days"] = days_between(
    oc["order_purchase_timestamp"], oc["order_estimated_delivery_date"]
)

# Delay: positive = arrived AFTER the estimated date.
oc["delay_days"] = days_between(
    oc["order_estimated_delivery_date"], oc["order_delivered_customer_date"]
)

# Stage durations, nulled out for the rows flagged in Cell 5.
oc["approval_to_carrier_days"] = days_between(
    oc["order_approved_at"], oc["order_delivered_carrier_date"]
).mask(oc["flag_carrier_before_approval"])

oc["carrier_to_customer_days"] = days_between(
    oc["order_delivered_carrier_date"], oc["order_delivered_customer_date"]
).mask(oc["flag_delivered_before_carrier"])

In [11]:
# An order is late if it arrived after the estimated delivery date.
# Only defined where a delivery date exists; otherwise left as missing.
has_delivery = oc["order_delivered_customer_date"].notna()
oc["is_late"] = (oc["delay_days"] > 0).where(has_delivery)

print(oc["is_late"].value_counts(dropna=False).to_string())

is_late
False    88649
True      7827
NaN       2965


In [12]:
delivered = oc["order_status"] == "delivered"

print("delivered orders:                   ", int(delivered.sum()))
print("delivered with a delivery date:     ", int((delivered & has_delivery).sum()))
print("delivery_days non-null (all orders):", int(oc["delivery_days"].notna().sum()))
print("delivery_days negative:             ", int((oc["delivery_days"] < 0).sum()))
print("approval_to_carrier_days non-null:  ", int(oc["approval_to_carrier_days"].notna().sum()))
print("carrier_to_customer_days non-null:  ", int(oc["carrier_to_customer_days"].notna().sum()))
print("is_late missing:                    ", int(oc["is_late"].isna().sum()))

delivered orders:                    96478
delivered with a delivery date:      96470
delivery_days non-null (all orders): 96476
delivery_days negative:              0
approval_to_carrier_days non-null:   96285
carrier_to_customer_days non-null:   96452
is_late missing:                     2965


In [13]:
# Orders that have a customer delivery date but are NOT status 'delivered'.
odd = has_delivery & ~delivered
print("has delivery date but status != delivered:", int(odd.sum()))
print(oc.loc[odd, "order_status"].value_counts().to_string())

has delivery date but status != delivered: 6
order_status
canceled    6


In [14]:
# Population rule (accepted): status is 'delivered' AND a delivery date exists.
oc["in_delivery_population"] = (
    (oc["order_status"] == "delivered")
    & oc["order_delivered_customer_date"].notna()
)

pop = oc[oc["in_delivery_population"]]
print("orders in population:         ", len(pop))
print("excluded orders:              ", len(oc) - len(pop))
print("is_late missing in population:", int(pop["is_late"].isna().sum()))
print("late orders in population:    ", int(pop["is_late"].sum()))
print(f"late rate in population:       {pop['is_late'].mean():.4f}")

orders in population:          96470
excluded orders:               2971
is_late missing in population: 0
late orders in population:     7826
late rate in population:       0.0811


In [15]:
# Percentiles show the typical value and the tails.
cols = ["delivery_days", "estimated_days", "delay_days"]
print(pop[cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2).to_string())

# Extremes, to see whether any value looks implausible.
print("\nLongest deliveries (days):")
print(pop["delivery_days"].nlargest(5).round(1).to_string())
print("\nLatest vs estimate (delay_days):")
print(pop["delay_days"].nlargest(5).round(1).to_string())

       delivery_days  estimated_days  delay_days
count       96470.00        96470.00    96470.00
mean           12.56           23.74      -11.18
std             9.55            8.76       10.18
min             0.53            2.01     -146.02
1%              1.83            6.05      -35.14
25%             6.77           18.33      -16.24
50%            10.22           23.23      -11.95
75%            15.72           28.41       -6.39
95%            29.27           38.42        3.82
99%            46.05           50.21       18.94
max           209.63          155.14      188.98

Longest deliveries (days):
19590    209.6
55619    208.4
61610    195.6
70307    194.9
89130    194.6

Latest vs estimate (delay_days):
55619    189.0
19590    181.6
11399    175.9
81401    167.7
89130    166.6


In [16]:
# Every estimated date appeared to be at 00:00:00. Confirm it.
est = pop["order_estimated_delivery_date"]
print("estimated dates all at midnight:", bool((est == est.dt.normalize()).all()))

# Strict definition (current): delivered timestamp after midnight of the estimated day.
late_strict = pop["delay_days"] > 0

# Calendar-day definition: late only if the delivery DATE is after the estimated DATE.
late_by_date = pop["order_delivered_customer_date"].dt.normalize() > est

print("\nlate (timestamp rule): ", int(late_strict.sum()))
print("late (calendar-day rule):", int(late_by_date.sum()))
print("orders that differ:      ", int((late_strict != late_by_date).sum()))
print(f"late rate, timestamp rule:    {late_strict.mean():.4f}")
print(f"late rate, calendar-day rule: {late_by_date.mean():.4f}")

estimated dates all at midnight: True

late (timestamp rule):  7826
late (calendar-day rule): 6534
orders that differ:       1292
late rate, timestamp rule:    0.0811
late rate, calendar-day rule: 0.0677


In [17]:
# Keep the old definition under a clear name for the sensitivity check.
oc["is_late_timestamp_rule"] = (oc["delay_days"] > 0).where(
    oc["order_delivered_customer_date"].notna()
)

# Final definition: late only if the delivery DATE is after the estimated DATE.
delivered_date = oc["order_delivered_customer_date"].dt.normalize()
oc["is_late"] = (delivered_date > oc["order_estimated_delivery_date"]).where(
    oc["order_delivered_customer_date"].notna()
)

# Check on the defined population.
pop = oc[oc["in_delivery_population"]]
print("is_late missing in population:", int(pop["is_late"].isna().sum()))
print("late (calendar-day):          ", int(pop["is_late"].sum()))
print("late (timestamp, sensitivity):", int(pop["is_late_timestamp_rule"].sum()))
print(f"late rate (calendar-day):      {pop['is_late'].mean():.4f}")

is_late missing in population: 0
late (calendar-day):           6534
late (timestamp, sensitivity): 7826
late rate (calendar-day):      0.0677


In [18]:
items = pd.read_csv(DATA_RAW / "olist_order_items_dataset.csv")
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

print("items loaded:", items.shape)
print("missing shipping_limit_date:", int(items["shipping_limit_date"].isna().sum()))
print("range:", items["shipping_limit_date"].min(), "to", items["shipping_limit_date"].max())

items loaded: (112650, 7)
missing shipping_limit_date: 0
range: 2016-09-19 00:15:34 to 2020-04-09 22:35:08


In [19]:
cols = ["price", "freight_value"]
print(items[cols].describe(percentiles=[0.01, 0.5, 0.95, 0.99]).round(2).to_string())

print("\nprice <= 0:        ", int((items["price"] <= 0).sum()))
print("freight_value == 0:", int((items["freight_value"] == 0).sum()))
print("freight_value < 0: ", int((items["freight_value"] < 0).sum()))

           price  freight_value
count  112650.00      112650.00
mean      120.65          19.99
std       183.63          15.81
min         0.85           0.00
1%          9.99           4.42
50%        74.99          16.26
95%       349.90          45.12
99%       890.00          84.52
max      6735.00         409.68

price <= 0:         0
freight_value == 0: 383
freight_value < 0:  0


In [20]:
# Join the deadline to the purchase time to see how far ahead each deadline is.
# validate="many_to_one": each item must match exactly one order.
chk = items.merge(
    orders_clean[["order_id", "order_purchase_timestamp"]],
    on="order_id", how="left", validate="many_to_one",
)
chk["limit_days"] = (
    chk["shipping_limit_date"] - chk["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(chk["limit_days"].describe(percentiles=[0.01, 0.5, 0.99]).round(2).to_string())
print("\ndeadline before purchase:", int((chk["limit_days"] < 0).sum()))
print("deadline > 30 days after purchase:", int((chk["limit_days"] > 30).sum()))
print("deadline > 365 days after purchase:", int((chk["limit_days"] > 365).sum()))

count    112650.00
mean          6.64
std           7.07
min           2.00
1%            2.01
50%           6.01
99%          19.01
max        1056.04

deadline before purchase: 0
deadline > 30 days after purchase: 239
deadline > 365 days after purchase: 4


In [21]:
long_limit = chk["limit_days"] > 30

# Are these items concentrated in a few sellers or orders?
print("items:  ", int(long_limit.sum()))
print("orders: ", chk.loc[long_limit, "order_id"].nunique())
print("sellers:", items.loc[long_limit, "seller_id"].nunique())

# Status of the affected orders.
affected = chk.loc[long_limit, "order_id"].unique()
print("\nStatus of affected orders:")
print(orders_clean.loc[orders_clean["order_id"].isin(affected), "order_status"]
      .value_counts().to_string())

items:   239
orders:  209
sellers: 45

Status of affected orders:
order_status
delivered     187
processing     10
canceled        7
shipped         3
invoiced        2


In [22]:
# Flag items whose shipping deadline is more than 30 days after purchase.
# The date itself is NOT edited. chk has the same row order as items
# (a many_to_one left join keeps the left table's order), so the
# boolean Series lines up row for row.
assert len(chk) == len(items)
assert (chk["order_id"].values == items["order_id"].values).all()

items["flag_long_shipping_limit"] = (chk["limit_days"] > 30).values

print("flagged items:", int(items["flag_long_shipping_limit"].sum()))
print("rows:", len(items))

flagged items: 239
rows: 112650


In [23]:
products = pd.read_csv(DATA_RAW / "olist_products_dataset.csv")
trans = pd.read_csv(DATA_RAW / "product_category_name_translation.csv")

# Classify each product's category status.
is_null = products["product_category_name"].isna()
in_trans = products["product_category_name"].isin(trans["product_category_name"])

products["category_status"] = "translated"
products.loc[is_null, "category_status"] = "null_category"
products.loc[~is_null & ~in_trans, "category_status"] = "missing_from_translation"

print(products["category_status"].value_counts().to_string())

category_status
translated                  32328
null_category                 610
missing_from_translation       13


In [24]:
# many_to_one: each item must match exactly one product, so rows cannot multiply.
items_cat = items.merge(
    products[["product_id", "category_status"]],
    on="product_id", how="left", validate="many_to_one",
)
assert len(items_cat) == len(items)

impact = items_cat.groupby("category_status").agg(
    items=("product_id", "size"),
    products=("product_id", "nunique"),
    revenue=("price", "sum"),
)
impact["pct_items"] = (impact["items"] / impact["items"].sum() * 100).round(2)
impact["pct_revenue"] = (impact["revenue"] / impact["revenue"].sum() * 100).round(2)
print(impact.round(2).to_string())

                           items  products      revenue  pct_items  pct_revenue
category_status                                                                
missing_from_translation      24        13      5514.48       0.02         0.04
null_category               1603       610    179535.28       1.42         1.32
translated                111023     32328  13406593.94      98.56        98.64


In [25]:
# Manual translations for the 2 categories missing from the translation table.
# These are my reading of the Portuguese names: check them yourself.
MANUAL_TRANSLATIONS = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers",
}

# Start from the official translation table.
translation_map = dict(zip(trans["product_category_name"],
                           trans["product_category_name_english"]))
translation_map.update(MANUAL_TRANSLATIONS)

# Map each product; null categories become 'unknown' (no guessing).
products["category_english"] = (
    products["product_category_name"].map(translation_map).fillna("unknown")
)

# Checks.
print("null category_english:", int(products["category_english"].isna().sum()))
print("'unknown' products:   ", int((products["category_english"] == "unknown").sum()))
print("distinct categories:  ", products["category_english"].nunique())
print(products.groupby("category_status")["category_english"].nunique().to_string())

null category_english: 0
'unknown' products:    610
distinct categories:   74
category_status
missing_from_translation     2
null_category                1
translated                  71


In [26]:
reviews = pd.read_csv(DATA_RAW / "olist_order_reviews_dataset.csv")
for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col])

print("reviews loaded:", reviews.shape)

# Orders with more than one review.
per_order = reviews.groupby("order_id").size()
multi = per_order[per_order > 1].index
print("orders with 2+ reviews:", len(multi))

# For those orders: do the reviews differ in score, and in time?
m = reviews[reviews["order_id"].isin(multi)]
g = m.groupby("order_id")
print("orders whose scores differ:        ", int((g["review_score"].nunique() > 1).sum()))
print("orders with identical creation date:", int((g["review_creation_date"].nunique() == 1).sum()))
print("orders with identical answer time:  ", int((g["review_answer_timestamp"].nunique() == 1).sum()))

reviews loaded: (99224, 7)
orders with 2+ reviews: 547
orders whose scores differ:         202
orders with identical creation date: 155
orders with identical answer time:   0


In [1]:
import unicodedata
from pathlib import Path
import pandas as pd

DATA_RAW = Path("..") / "data" / "raw"
customers = pd.read_csv(DATA_RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(DATA_RAW / "olist_sellers_dataset.csv")

def normalize(text):
    """Lowercase, trim, and strip accents so spelling variants group together."""
    text = unicodedata.normalize("NFKD", str(text).strip().lower())
    return "".join(c for c in text if not unicodedata.combining(c))

for name, df, city_col, state_col in [
    ("customers", customers, "customer_city", "customer_state"),
    ("sellers",   sellers,   "seller_city",   "seller_state"),
]:
    print(f"=== {name} ===")
    states = df[state_col]
    print("distinct states:", states.nunique())
    print("states not 2 uppercase letters:", int((~states.str.fullmatch(r"[A-Z]{2}")).sum()))

    raw_n = df[city_col].nunique()
    norm = df[city_col].map(normalize)
    print("distinct cities raw:       ", raw_n)
    print("distinct cities normalized:", norm.nunique())

    # Cities with characters other than letters, spaces, apostrophes, hyphens.
    odd = df.loc[~norm.str.fullmatch(r"[a-z '\-]+"), city_col]
    print("cities with unusual characters:", odd.nunique())
    print("examples:", odd.drop_duplicates().head(8).tolist())

    # Is any city string both padded and unpadded, or differently cased?
    print("cities with leading/trailing spaces:",
          int((df[city_col] != df[city_col].str.strip()).sum()))
    print()

=== customers ===
distinct states: 27
states not 2 uppercase letters: 0
distinct cities raw:        4119
distinct cities normalized: 4119
cities with unusual characters: 1
examples: ['quilometro 14 do mutum']
cities with leading/trailing spaces: 0

=== sellers ===
distinct states: 23
states not 2 uppercase letters: 0
distinct cities raw:        611
distinct cities normalized: 609
cities with unusual characters: 21
examples: ['auriflama/sp', 'sao paulo / sao paulo', '04482255', 'novo hamburgo, rio grande do sul, brasil', 'cariacica / es', 'sbc/sp', "arraial d'ajuda (porto seguro)", 'santo andre/sao paulo']
cities with leading/trailing spaces: 0



In [2]:
products = pd.read_csv(DATA_RAW / "olist_products_dataset.csv")
size_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print(products[size_cols].describe(percentiles=[0.01, 0.5, 0.99]).round(1).to_string())
print("\nweight == 0:", int((products["product_weight_g"] == 0).sum()))
for c in size_cols[1:]:
    print(f"{c} == 0:", int((products[c] == 0).sum()))

items_clean = pd.read_csv(Path("..") / "data" / "processed" / "order_items_clean.csv")
print("\nTop 5 items by price:")
print(items_clean.nlargest(5, "price")[["price", "freight_value", "category_english"]].to_string())

       product_weight_g  product_length_cm  product_height_cm  product_width_cm
count           32949.0            32949.0            32949.0           32949.0
mean             2276.5               30.8               16.9              23.2
std              4282.0               16.9               13.6              12.1
min                 0.0                7.0                2.0               6.0
1%                 60.0               16.0                2.0              11.0
50%               700.0               25.0               13.0              20.0
99%             22538.0              100.0               69.0              63.0
max             40425.0              105.0              105.0             118.0

weight == 0: 4
product_length_cm == 0: 0
product_height_cm == 0: 0
product_width_cm == 0: 0

Top 5 items by price:
         price  freight_value  category_english
3556    6735.0         194.31        housewares
112233  6729.0         193.21         computers
107841  6499.0      

In [3]:
norm = sellers["seller_city"].map(normalize)
odd_mask = ~norm.str.fullmatch(r"[a-z '\-]+")
print(sellers[odd_mask].groupby("seller_city").size().sort_values(ascending=False).to_string())

# The 2 variants that merge after normalization:
g = sellers.assign(n=norm).groupby("n")["seller_city"].nunique()
print(sellers[sellers["seller_city"].map(normalize).isin(g[g > 1].index)]
      [["seller_city"]].drop_duplicates().sort_values("seller_city").to_string())

seller_city
04482255                                    1
arraial d'ajuda (porto seguro)              1
auriflama/sp                                1
barbacena/ minas gerais                     1
carapicuiba / sao paulo                     1
cariacica / es                              1
jacarei / sao paulo                         1
maua/sao paulo                              1
mogi das cruzes / sp                        1
novo hamburgo, rio grande do sul, brasil    1
pinhais/pr                                  1
ribeirao preto / sao paulo                  1
rio de janeiro / rio de janeiro             1
rio de janeiro \rio de janeiro              1
rio de janeiro, rio de janeiro, brasil      1
santo andre/sao paulo                       1
sao paulo / sao paulo                       1
sao sebastiao da grama/sp                   1
sbc/sp                                      1
sp / sp                                     1
vendas@creditparts.com.br                   1
               seller_

In [4]:
zero_w = products.loc[products["product_weight_g"] == 0, "product_id"]
print("items affected:", int(items_clean["product_id"].isin(zero_w).sum()))

items affected: 8


In [5]:
import pandas as pd
s = pd.read_csv("../data/processed/sellers_clean.csv")
mask = s["seller_city_clean"].str.contains(r"\s[a-z]{2}$", na=False)
print(s.loc[mask, ["seller_city", "seller_city_clean", "seller_state"]]
        .drop_duplicates().to_string())

            seller_city  seller_city_clean seller_state
78           lages - sc         lages - sc           SC
795         brasilia df        brasilia df           DF
826      sao paulo - sp     sao paulo - sp           SP
1644       sao paulo sp       sao paulo sp           SP
1686  angra dos reis rj  angra dos reis rj           RJ
2589    aguas claras df    aguas claras df           SP
